In [10]:
import json
import random

RUNTIME_OUT_DIR = '../mock-data/'

NAMES = 'francesco giammarco marco giovanni pino gianluca cristian lucrezio antonio carmeloriccardo anastasio luca simone giorgia francesca ilenia simona virginia vanessa martina giovanna maria veronica matilde irene'
NAMES = NAMES.split(' ')

SURNAMES = 'marzano tocco pollara parisi butera jackson gerardi esposito russo ricci moretti marino greco marino barbieri cassano caruso ferri pellegrini conte gatti'
SURNAMES = SURNAMES.split(' ')

ADDRESSES = 'gmail hotmail microsoft unipa edu'
ADDRESSES = ADDRESSES.split(' ')

DOMAINS = 'com net it ru eu'
DOMAINS = DOMAINS.split(' ')

NUMBERS = [n for n in range(1, 100)]

USERNAMES = 'gianpiero gianfrancesco giancazzo gianpisello gianmarrone gianpuzza gianmalato giangian giangiangian'
USERNAMES = USERNAMES.split(' ')


def compile(table, *peek):
  path = RUNTIME_OUT_DIR + table['name'] + '.json'
  rules = table['rules']
  primary_keys = table.get('primary', [])

  if not isinstance(primary_keys, list):
    raise TypeError("primary_keys must be a list or primary keys (strings)")

  def pick(arrays):
    return (random.choice(x) for x in arrays)

  def transform(value):
    fmt = value['fmt']
    values = pick(value['args'])
    return fmt.format(*values)

  objects = []
  uniques = set()

  max_count = 10000
  max_attempts = max_count * 20
  attempts = 0

  while len(objects) < max_count and attempts < max_attempts:
    attempts += 1
    candidate = {
      key: transform(value) for (key, value) in rules.items()
    }

    if primary_keys:
      values = tuple(candidate[key] for key in primary_keys)
      if values in uniques: 
        continue

      uniques.add(values)

    objects.append(candidate)

  if len(objects) < max_count:
    print("Warning: target amount was {} but {} where generated".format(
      max_count, 
      len(objects)
    ))

  for f in peek: f(objects)
  with open(path,  'w') as file:
    json.dump(objects, file)



ACCOUNT_TABLE = {
  'name': 'Account',
  'primary': ['email'],
  'rules':  {
    'email': {
      'fmt': '{}.{}{}@{}.{}',
      'args': [
        NAMES,
        SURNAMES,
        NUMBERS,
        ADDRESSES,
        DOMAINS
      ],
    },
    'password' : {
      'fmt': 'Password123',
      'args': []
    },
    'username': {
      'fmt': '{}',
      'args': [
        USERNAMES
      ]
    }
  }
}


def compile_utente_generico(accounts):
  objects = [
    {
      'account': account['email'],
      'utente_giocatore': f'(giocatore): {account['email']}',
      'utente_dungeon_master': f'(dungeon_master): {account['email']}'
    } for account in accounts
  ]

  with open(RUNTIME_OUT_DIR + 'UtenteGenerico.json', 'w+') as file:
    json.dump(objects, file)

def compile_amministratore(accounts):
  objects = [
    {
      'account': account['email'],
    } for account in accounts
  ]

  with open(RUNTIME_OUT_DIR + 'Amministratore.json', 'w+') as file:
    json.dump(objects, file)

compile(
  ACCOUNT_TABLE, 
  compile_amministratore,
  compile_utente_generico,
)
